In [ ]:
import sys
import copy
import torch
import numpy as np
import tensorflow as tf
import scipy as sp
import pandas as pd
import dill as pickle
import matplotlib.pyplot as plt

In [ ]:
sys.path.append("/u/zanardi3/Codes/ML/ROMAr/romar/")
from romar import env
from romar import const
from romar import utils
from romar import backend as bkd
from romar.systems import BoxAd

import romar.postproc.plotting as pltt

In [ ]:
sys.path.append("/u/zanardi3/Codes/ML/PyCOMET/pycomet/")
from pycomet.networks.chemkin.plasma_cr.meno.blocks import PlasmaCROperators
from pycomet.networks.chemkin.plasma_cr.meno import const

In [ ]:
env_opts = {
  "backend": "numpy",
  "device": "cpu",
  "device_idx": 0,
  "nb_threads": 2,
  "epsilon": 1e-16,
  "floatx": "float64"
}
env.set(**env_opts)

In [ ]:
path_to_dtb = "/u/zanardi3/Codes/ML/ROMAr/romar/examples/database"

In [ ]:
system = BoxAd(
  species={k: path_to_dtb + "/species/" + k + ".json" for k in ("Ar", "Arp", "em")},
  kin_dtb=path_to_dtb + "/rates/kin_fit.p",
  rad_dtb=path_to_dtb + "/rates/rad_fit.p",
  use_rad=True
)

In [ ]:
rho = 1e-2
T = 3.5e4
Te = 3e2
quad_mu = {
  "x": np.array([rho, T, Te]).reshape(1,-1),
  "w": np.ones((1,1))
}
tlim = [1e-12, 1e-2]

In [ ]:
t = np.geomspace(*tlim, 200)
y0, rho = system.equil.get_init_sol(quad_mu["x"].squeeze())
y0

In [ ]:
# with open("./yfom.p", "rb") as file:
#   yfom = pickle.load(file)
# # Setting up
# system.use_rom = False
# y0 = system.set_up(y0, rho)

In [ ]:
yfom, _ = system.solve_fom(t, y0, rho)

In [ ]:
w_fom = copy.deepcopy(yfom[:system.mix.nb_comp])
n_fom = system.mix.get_n(bkd.to_torch(w_fom)).numpy()
T_fom = copy.deepcopy(yfom[system.mix.nb_comp:])
T_fom[-1] = system.mix.get_Te(pe=T_fom[-1], ne=n_fom[0])

In [ ]:
for i in range(2):
  plt.semilogx(t, T_fom[i])
plt.show()
plt.close()

In [ ]:
for s in system.mix.species.values():
  print(s.name)
  for i in s.indices:
    plt.loglog(t, n_fom[i])
  plt.show()
  plt.close()

In [ ]:
species = ("em", "Ar", "Arp")
path_to_dtb = "/u/zanardi3/Workspace/pycomet/plasma_cr/00.datagen.r01/database/"
model = PlasmaCROperators(
  species={k: path_to_dtb + "/species/" + k + ".json" for k in species},
  rates=path_to_dtb + "/rates/rates.hdf5",
  use_corr=False,
  scale_corr=False
)

In [ ]:
mix = model.mix["h"]

In [ ]:
w = w_fom.T
Th = T_fom[0].reshape(-1,1)
Te = T_fom[1].reshape(-1,1)

In [ ]:
e = mix._e(w, Th, Te)
Te2 = mix._Te(w, Th, e[:1])

In [ ]:
plt.semilogx(t, e)

In [ ]:
i = t < 1e-3
plt.semilogx(t[i], Te[i])
plt.semilogx(t[i], Te2[i])

In [ ]:
stop

Rates

In [ ]:
def get_rates_tf(Th, Te):
  Th = Th.reshape(1,1)
  Te = Te.reshape(1,1)
  return model.compute_rates(Th, Te, f=None)

def get_rates_np(Th, Te):
  Th = bkd.to_torch(Th)
  Te = torch.clip(bkd.to_torch(Te), const.TMIN, const.TMAX)
  system.mix.update_species_thermo(Th, Te)
  system.kin.update(Th, Te)
  system.rad.update(Th, Te)
  return {**system.sources.kin.rates, **system.sources.rad.rates}

In [ ]:
Th, Te = T_fom
for _ in range(10):
  i = np.random.choice(np.arange(200))
  Thi, Tei = Th[i], Te[i]
  rates_tf = get_rates_tf(Thi, Tei)
  rates_np = get_rates_np(Thi, Tei)
  for k1 in ("bwd", "fwd"):
    for k2 in ("EXe", "EXh", "BB"):
      if k1 == "fwd" and k2 == "BB":
        continue
      rtf = rates_tf[k1][k2].numpy().squeeze()
      rnp = rates_np[k2][k1].numpy()
      if k1 == "bwd":
        rnp = rnp.T
      rnp = rnp[rnp>0.0]
      print(i, k1, k2, (np.abs(rnp - rtf) / (rnp+1e-80)).max())

In [ ]:
Th, Te = T_fom
for _ in range(10):
  i = np.random.choice(np.arange(200))
  Thi, Tei = Th[i], Te[i]
  rates_tf = get_rates_tf(Thi, Tei)
  rates_np = get_rates_np(Thi, Tei)
  for k1 in ("bwd", "fwd"):
    for k2 in ("Ie", "Ih", "BF"):
      if k1 == "fwd" and k2 == "BF":
        continue
      rtf = rates_tf[k1][k2].numpy().squeeze()
      rnp = rates_np[k2][k1].numpy()
      if k1 == "bwd":
        rnp = rnp.T
      rnp = rnp[rnp>0.0]
      print(i, k1, k2, (np.abs(rnp - rtf) / (rnp+1e-80)).max())

In [ ]:
def get_rhs_tf(x):
  x = tf.convert_to_tensor(x)
  y = _get_rhs_tf(x)
  return y.numpy().squeeze()

def _get_rhs_tf(x):
  n_em, n_ar1, xlin, Th, Te = tf.split(x, [1,1,32,1,1], axis=-1)
  A, b = model((n_em, n_ar1, Th, Te, None))
  xlin = tf.expand_dims(xlin, axis=-1)
  return tf.squeeze(tf.matmul(A, xlin)) + b

def get_rhs_np(x):
  dx = []
  x = bkd.to_torch(x)
  for xi in x:
    n, Th, Te = xi[:-2], xi[-2], xi[-1]
    Th = torch.clip(Th, const.TMIN, const.TMAX)
    Te = torch.clip(Te, const.TMIN, const.TMAX)
    f_rho, *_ = system.sources.call_ad(n, Th, Te)
    f_n = system.mix.m_inv_mat @ f_rho
    dx.append(f_n[2:])
  return np.vstack(dx)

In [ ]:
x = np.vstack([n_fom, T_fom]).T
# i = 100
dx_tf = get_rhs_tf(x)
dx_np = get_rhs_np(x)
err = np.abs(dx_np-dx_tf)/(np.abs(dx_np)+1e-50)

In [ ]:
dx_np, dx_tf, err

In [ ]:
i = np.argmax(err.reshape(-1))
dx_np.reshape(-1)[i], dx_tf.reshape(-1)[i], err.max()

In [ ]:
i = np.random.choice(np.arange(200))
xi = tf.convert_to_tensor(x[i:i+1])

with tf.GradientTape() as tape:
  tape.watch(xi)
  y = _get_rhs_tf(xi)

jacobian = tape.jacobian(y, xi).numpy().squeeze()

print(jacobian)
# tf.Tensor(
# [[2. 1.]
#  [2. 1.]], shape=(2, 2), dtype=float32)